# USPS digits — §4.3 / figs. 6–8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/03_usps_digits.ipynb)

Port of Matlab [`demUsps.m`](https://github.com/SheffieldML/deepGP/blob/master/deepGP/matlab/demUsps.m) /
[`demDigitsDemonstration.m`](https://github.com/SheffieldML/deepGP/blob/master/deepGP/matlab/demDigitsDemonstration.m):
`vercat2` (50×{0,6,1}), depth-5 widths Q={15,12,10,8,6}, K=80.

Metrics: variational bound and leave-one-out 1-NN error in the top latent space.


## Setup


In [ ]:
import sys, subprocess, os, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd().resolve()
# Prefer sibling of notebooks/ (repo layout)
for c in [ROOT.parent, ROOT, ROOT / "examples" / "damianou2013"]:
    if (c / "_common.py").is_file():
        ROOT = c
        break

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "numpy", "scipy", "matplotlib", "GPy>=1.15", "scikit-learn"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        f"git+https://github.com/SheffieldML/PyDeepGP.git@examples-damianou2013"])
    ROOT = Path.cwd().resolve()
    (ROOT / "data").mkdir(exist_ok=True)
    for dest, url in [
        ("_common.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_common.py"),
        ("_matlab_data.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_matlab_data.py"),
        ("data/hgplvmSampleDataTr1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/hgplvmSampleDataTr1.mat"),
        ("data/demHighFiveHgplvm1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/demHighFiveHgplvm1.mat"),
    ]:
        p = ROOT / dest
        if not p.is_file():
            print("Fetching", dest)
            urllib.request.urlretrieve(url, str(p))

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working directory:", ROOT)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import deepgp
from _common import (
    ard_weights, dominant_dims, nearest_neighbour_error, optimize_staged, print_ard, rbf_bias,
)
from _matlab_data import USPS_DEPTH_Q, load_usps_vercat2


In [ ]:
QUICK = True  # True → fewer digits / smaller Q / depths 1–2
DEPTHS = [1, 2] if QUICK else [1, 2, 3, 4, 5]
per_class = 20 if QUICK else 50
num_inducing = 25 if QUICK else 80
iters_fixed = 40 if QUICK else 500
iters_free = 100 if QUICK else 3000


## Load USPS `vercat2` subset


In [ ]:
Y, labels = load_usps_vercat2(per_class=per_class)
Y = Y - Y.mean(0)
Y = Y / (Y.std() + 1e-12)
print(f"Y={Y.shape}, classes={sorted(set(labels.tolist()))}")


## Fit DeepGPs at increasing depth


In [ ]:
def depth_dimensions(depth, data_dim):
    return [data_dim] + list(USPS_DEPTH_Q[depth])

results = []
models = {}
for depth in DEPTHS:
    dims = depth_dimensions(depth, Y.shape[1])
    if QUICK:
        dims = [Y.shape[1]] + [max(2, q // 2) for q in dims[1:]]
    print(f"\n=== depth {depth}  dims={dims} ===")
    m = deepgp.DeepGP(
        dims, Y,
        kernels=[rbf_bias(q) for q in dims[1:]],
        num_inducing=num_inducing,
        back_constraint=False,
        inits="PCA",
    )
    optimize_staged(m, iters_fixed, iters_free, messages=False)
    bound = float(m.log_likelihood())
    Xtop = np.asarray(m.layers[-1].X.mean)
    mistakes, rate = nearest_neighbour_error(Xtop, labels)
    print(f"bound={bound:.3f}  1-NN mistakes={mistakes}/{Y.shape[0]} ({rate:.3%})")
    print_ard(m)
    results.append((depth, bound, mistakes, rate))
    models[depth] = m

print("\ndepth  bound      1-NN errors")
for depth, bound, mistakes, rate in results:
    print(f"{depth:5d}  {bound:9.3f}  {mistakes:3d} ({rate:.3%})")


## Top latent scatter and ARD (figs. 6–8 style)


In [ ]:
deepest = max(DEPTHS)
m = models[deepest]
Xtop = np.asarray(m.layers[-1].X.mean)
dims_plot = dominant_dims(ard_weights(m.layers[-1].kern), k=2)
colors = {0: "C0", 1: "C1", 6: "C2"}
markers = {0: "o", 1: "s", 6: "^"}

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for d in (0, 1, 6):
    mask = labels == d
    ax.scatter(
        Xtop[mask, dims_plot[0]], Xtop[mask, dims_plot[1]],
        c=colors[d], marker=markers[d], s=28, label=str(d),
    )
ax.set_title(f"USPS top latent (depth {deepest})")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(m.layers), figsize=(3.0 * len(m.layers), 3))
if len(m.layers) == 1:
    axes = [axes]
for i, layer in enumerate(m.layers):
    w = ard_weights(layer.kern)
    axes[i].bar(np.arange(1, len(w) + 1), w)
    axes[i].set_title(f"layer {i}")
fig.suptitle("ARD (fig. 6 style)")
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot([r[0] for r in results], [r[1] for r in results], "o-", label="bound")
ax.set_xlabel("depth")
ax.set_ylabel("variational bound")
ax2 = ax.twinx()
ax2.plot([r[0] for r in results], [r[2] for r in results], "s--", color="C1")
ax2.set_ylabel("1-NN mistakes")
ax.set_title("Depth vs bound / 1-NN")
fig.tight_layout()
plt.show()
